In [1]:
import sagemaker
import boto3
import os
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()

bucket = session.default_bucket()
prefix = "tarea-6-processing-byoc"

print("Region:", region)
print("Role:", role)
print("Bucket:", bucket)
print("Prefix:", prefix)

Region: us-east-1
Role: arn:aws:iam::995371347105:role/SageMakerStudioExecutionRole2026
Bucket: sagemaker-us-east-1-995371347105
Prefix: tarea-6-processing-byoc


In [3]:
project_root = os.path.abspath(".")

local_data_path = os.path.join(project_root, "data", "raw", "sales_train.csv")
script_path = os.path.join(project_root, "processing", "prep.py")
dockerfile_dir = os.path.join(project_root, "processing", "container")

print(local_data_path)
print(script_path)
print(dockerfile_dir)

/home/sagemaker-user/Tarea-6-Equipo-2/Tarea_6/data/raw/sales_train.csv
/home/sagemaker-user/Tarea-6-Equipo-2/Tarea_6/processing/prep.py
/home/sagemaker-user/Tarea-6-Equipo-2/Tarea_6/processing/container


In [4]:
s3_input_data = session.upload_data(
    path=local_data_path,
    bucket=bucket,
    key_prefix=f"{prefix}/input"
)

print("S3 input path:", s3_input_data)

S3 input path: s3://sagemaker-us-east-1-995371347105/tarea-6-processing-byoc/input/sales_train.csv


In [5]:
import subprocess

account_id = boto3.client("sts").get_caller_identity()["Account"]
repository_name = "tarea-6-processing-byoc"
image_tag = "latest"

image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repository_name}:{image_tag}"

print("Account ID:", account_id)
print("Image URI:", image_uri)

Account ID: 995371347105
Image URI: 995371347105.dkr.ecr.us-east-1.amazonaws.com/tarea-6-processing-byoc:latest


In [6]:
ecr = boto3.client("ecr")

try:
    ecr.describe_repositories(repositoryNames=[repository_name])
    print(f"El repositorio {repository_name} ya existe.")
except ecr.exceptions.RepositoryNotFoundException:
    ecr.create_repository(repositoryName=repository_name)
    print(f"Repositorio {repository_name} creado.")

Repositorio tarea-6-processing-byoc creado.


In [7]:
login_password = boto3.client("ecr").get_authorization_token()["authorizationData"][0]["authorizationToken"]

print("Listo para hacer login a ECR desde terminal o notebook.")

Listo para hacer login a ECR desde terminal o notebook.


In [8]:
s3_output_path = f"s3://{bucket}/{prefix}/output/"
print("S3 output path:", s3_output_path)

S3 output path: s3://sagemaker-us-east-1-995371347105/tarea-6-processing-byoc/output/


In [9]:
script_processor = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=session
)

In [10]:
script_processor.run(
    code=script_path,
    inputs=[
        ProcessingInput(
            source=s3_input_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=s3_output_path
        )
    ]
)

INFO:sagemaker:Creating processing-job with name tarea-6-processing-byoc-2026-03-17-02-46-52-578


........2026-03-17 02:48:05,009 - __main__ - INFO - Iniciando Carga de Datos...
2026-03-17 02:48:06,760 - __main__ - INFO - Tiempo de ejecución: 1.75 segundos
2026-03-17 02:48:06,760 - __main__ - INFO - Guardando las ventas mensuales...
Guardado: /opt/ml/processing/output/monthly_sales.csv (shape=(1609124, 4))



In [11]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=f"{prefix}/output"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

tarea-6-processing-byoc/output/monthly_sales.csv


In [15]:
import pandas as pd

output_s3_uri = f"s3://{bucket}/{prefix}/output/monthly_sales.csv"
df_out = pd.read_csv(output_s3_uri)

df_out.head()

,date_block_num,shop_id,item_id,item_cnt_month
0,0,0,32,6.0
1,0,0,33,3.0
2,0,0,35,1.0
3,0,0,43,1.0
4,0,0,51,2.0


In [13]:
df_out.shape

(1609124, 4)